In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

# Fix dataset mirror issue if the official server is down
torchvision.datasets.CIFAR10.url = "https://data.brainchip.com/dataset-mirror/cifar10/cifar-10-python.tar.gz"

In [ ]:
# 1. TRANSFORMS: Define different pipelines for Training vs Testing
# Training pipeline with Data Augmentation for regularizing the model
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),    # 50% chance to flip the image left-to-right
    transforms.RandomCrop(32, padding=4),      # Pad the edges, then take a random 32x32 crop
    transforms.ToTensor(),                     # Convert pixels (0-255) to PyTorch Tensors (0.0-1.0)
    transforms.Normalize((0.5, 0.5, 0.5), 
                         (0.5, 0.5, 0.5))      # Normalize to standard [-1, 1] range for stability
])

# Testing pipeline (No augmentation, pristine images only)
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) 
])

batch_size = 64

# 2. DATASETS: Fetch the CIFAR-10 dataset
trainset = torchvision.datasets.CIFAR10(root='./Data', train=True, download=True, transform=train_transform)
testset = torchvision.datasets.CIFAR10(root='./Data', train=False, download=True, transform=test_transform)

# 3. DATALOADERS: Handle batching and shuffling
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
print(f"Data pipeline ready! Number of training batches: {len(trainloader)}")

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Convolutional Block 1
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=5, padding=2)
        self.max_pool1 = nn.MaxPool2d(kernel_size=2)
        
        # Convolutional Block 2
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.max_pool2 = nn.MaxPool2d(kernel_size=2)
        
        # Convolutional Block 3
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.max_pool3 = nn.MaxPool2d(kernel_size=2)

        # Fully Connected Layers
        self.linear1 = nn.Linear(in_features=64 * 4 * 4, out_features=512)
        self.linear2 = nn.Linear(in_features=512, out_features=256)
        self.linear3 = nn.Linear(in_features=256, out_features=10)

        # Activation Function
        self.relu = nn.ReLU()

    def forward(self, x):
        # Feature extraction
        x = self.max_pool1(self.relu(self.conv1(x)))
        x = self.max_pool2(self.relu(self.conv2(x)))
        x = self.max_pool3(self.relu(self.conv3(x)))
        
        # Flattening
        x = x.view(x.shape[0], -1)
        
        # Classification
        x = self.relu(self.linear1(x))
        x = self.relu(self.linear2(x))
        x = self.linear3(x)

        return x

# Verify model output shape
model = SimpleCNN()
dummy_x = torch.randn(1, 3, 32, 32)
print("Output shape:", model(dummy_x).shape)

In [ ]:
# Hardware acceleration setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleCNN().to(device)

# Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 20
print(f"Training on {device}...")

for epoch in range(epochs):
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data[0].to(device), data[1].to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        
    print(f'Epoch {epoch + 1:02d}/{epochs} | Loss: {running_loss / len(trainloader):.4f}')

print('Finished Training!')

In [ ]:
correct = 0
total = 0

# Evaluation mode (no gradients needed)
with torch.no_grad():
    for data in testloader:
        images, labels = data[0].to(device), data[1].to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Model Training Accuracy: {accuracy:.2f}%')